# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  
**Name:** Simi Chakravarty

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [9]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [10]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [11]:
cleaned_df = df.drop_duplicates().copy()
print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)

log('drop duplicates', 'dropped all duplicate rows', 15)

Original shape: (315, 5)
Cleaned shape: (300, 5)
[drop duplicates] dropped all duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [12]:
cleaned_df['price'] = cleaned_df['price'].str.replace('$', '').astype(float)

# Confirm that price became a float data type
cleaned_df.dtypes

log('clean price', 'converted price to float', 0)

[clean price] converted price to float (0 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [17]:
cleaned_df['qty'] = pd.to_numeric(cleaned_df['qty'], errors='coerce')
cleaned_df = cleaned_df.dropna(subset=['qty'])
cleaned_df = cleaned_df[cleaned_df['qty'] > 0]
cleaned_df.shape

log('clean qty', 'dropped missing and negative values', 25)

[clean qty] dropped missing and negative values (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [21]:
print(cleaned_df['item'].value_counts())
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
}

cleaned_df['item'] = cleaned_df['item'].map(ITEM_MAP)
cleaned_df.shape

log('clean item', 'converted all items to standard in ITEM_MAP', 0)

item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


(275, 5)

### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [23]:
print(cleaned_df['category'].value_counts())
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Apparel',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear',
}

cleaned_df['category'] = cleaned_df['category'].map(CATEGORY_MAP)
cleaned_df.shape

log('clean category', 'converted all categories to standard in CATEGORY_MAP', 0)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[clean category] converted all categories to standard in CATEGORY_MAP (0 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [31]:
assert cleaned_df.duplicated().sum() == 0
assert cleaned_df['qty'].min() >= 1
assert cleaned_df['price'].dtype == float

allowed_items = ['Cheeseburger', 'Foam Finger', 'Rain Poncho']
assert set(cleaned_df['item'].unique()) == set(allowed_items)

allowed_categories = ['Food', 'Merch', 'Apparel', 'RainGear']
assert set(cleaned_df['category'].unique()) == set(allowed_categories)

print('clean:', df.shape)

clean: (315, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [ ]:
# TODO

**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [ ]:
import pandas as pd
pd.DataFrame(DECISIONS)

### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

_your answer here_